In [1]:
import googlemaps
import pandas as pd
from tqdm import tqdm
tqdm.pandas()
import gspread
from google.oauth2.service_account import Credentials
import json
import os
from dotenv import load_dotenv
from gspread_dataframe import set_with_dataframe

load_dotenv()

True

In [2]:
sheet_key = os.environ["SHEET_KEY"]
api_key = os.environ["GOOGLE_MAPS_API_KEY"]

SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive"
]

In [3]:
# Read data
creds = Credentials.from_service_account_info(json.loads(os.environ["GOOGLE_CREDENTIALS"]), scopes=SCOPES)
client = gspread.authorize(creds)

sheet = client.open_by_key(sheet_key).worksheet("customers")

data = sheet.get_all_records()
df = pd.DataFrame(data)

In [23]:
result = gmaps.geocode("ICA Supermarket Sollebrunn" + ", Sweden")



In [19]:
result[0].keys()

dict_keys(['address_components', 'formatted_address', 'geometry', 'navigation_points', 'partial_match', 'place_id', 'plus_code', 'types'])

In [21]:
result[0]["address_components"][4]["short_name"]

'Västra Götalands län'

In [24]:
result

[{'address_components': [{'long_name': '6',
    'short_name': '6',
    'types': ['street_number']},
   {'long_name': 'Trollhättevägen',
    'short_name': 'Trollhättevägen',
    'types': ['route']},
   {'long_name': 'Sollebrunn',
    'short_name': 'Sollebrunn',
    'types': ['postal_town']},
   {'long_name': 'Västra Götalands län',
    'short_name': 'Västra Götalands län',
    'types': ['administrative_area_level_1', 'political']},
   {'long_name': 'Sweden',
    'short_name': 'SE',
    'types': ['country', 'political']},
   {'long_name': '441 70', 'short_name': '441 70', 'types': ['postal_code']}],
  'formatted_address': 'Trollhättevägen 6, 441 70 Sollebrunn, Sweden',
  'geometry': {'location': {'lat': 58.12009489999999, 'lng': 12.5321491},
   'location_type': 'ROOFTOP',
   'viewport': {'northeast': {'lat': 58.12146013029151,
     'lng': 12.5333325802915},
    'southwest': {'lat': 58.1187621697085, 'lng': 12.5306346197085}}},
  'navigation_points': [{'location': {'latitude': 58.1201403,

In [26]:
#Add geo information
gmaps = googlemaps.Client(key=api_key)

def get_location_info(store_name):
    result = gmaps.geocode(store_name + ", Sweden")
    if not result:
        return None, None, None, None, None, None
    
    r = result[0]
    lat = r["geometry"]["location"]["lat"]
    lng = r["geometry"]["location"]["lng"]

    components = r["address_components"]
    street = next((c["long_name"] for c in components if "route" in c["types"]), None)
    number = next((c["long_name"] for c in components if "street_number" in c["types"]), None)
    postal = next((c["long_name"] for c in components if "postal_code" in c["types"]), None)
    city = next((c["long_name"] for c in components if "postal_town" in c["types"]), None)
    region = next((c["long_name"] for c in components if "administrative_area_level_1" in c["types"]), None)
    
    return city, street, number, postal, region, lat, lng

df["customer"].head(10).progress_apply(lambda x: pd.Series(get_location_info(x)))

  0%|          | 0/10 [00:00<?, ?it/s]

100%|██████████| 10/10 [00:01<00:00,  8.89it/s]


,0,1,2,3,4,5,6
0,Nödinge,Ale Torg,7,449 31,Västra Götalands län,57.894492,12.046865
1,Alvhem,Adolf Fredriks väg,14,446 91,Västra Götalands län,58.006670,12.159980
2,NaN,NaN,NaN,NaN,Västra Götaland County,57.930020,12.536211
3,Alingsås,Hemvägen,19,441 39,Västra Götalands län,57.924454,12.544616
4,Alingsås,Rubingatan,1,441 45,Västra Götalands län,57.920054,12.511940
5,Alingsås,Stationsgatan,30,441 30,Västra Götalands län,57.927399,12.536204
6,Alingsås,Noltorp centrum,NaN,441 15,Västra Götalands län,57.937829,12.521745
7,Sollebrunn,Trollhättevägen,6,441 70,Västra Götalands län,58.120095,12.532149
8,Almunge,Almungevägen,218,741 72,Uppsala län,59.875788,18.046287
9,Alsterbro,Alstervägen,33,382 73,Kalmar län,56.941624,15.914055


In [ ]:

#save data to new worksheet
try:
    worksheet = sheet.add_worksheet(title="customer_enriched", rows=1000, cols=20)
except:
    worksheet = sheet.worksheet("customer_enriched") 

set_with_dataframe(worksheet, df)